# UMUD R20 — dlaczego PA nie poprawia wyniku?

**Najlepsze znane zgłoszenie: 0,39776 = PA080 + FL refit + MT R12C c1.**
FL develop uzyskało 0,41433. Zostawiamy zwycięskie kolumny i sprawdzamy mechanizm PA.

Uruchom w **Google Colab → GPU T4 → Uruchom wszystko**. Wymagane pliki projektu
pozostają w dotychczasowych katalogach Drive. Notebook zawiera kod i manifesty;
nie potrzebujesz oddzielnego ZIP-u. Nie trenuje modeli i nie wysyła nic do Kaggle.

Przebieg:
1. Kontrola hashy wag, obrazów i syntetycznej geometrii.
2. Zamrożony refit: 309 test + 35 OSF; pięć definicji PA na tych samych maskach.
3. Zamrożony develop: orientacja masek GT w foldach 0 i 4 (824 unikalne obrazy).
4. Raport + nakładki + `UMUD_R20_WYNIKI.zip`, zapisywane na Drive.

Wznowienie: ponownie uruchom wszystkie komórki. Zapis per obraz, klucze obejmujące
wagi, obraz, progi, kod i środowisko; gotowe wyniki nie wymagają powtórzenia inferencji.
Nie uruchamiaj dwóch kopii jednocześnie w tym samym katalogu wyników.

OSF jest wielokrotnie używaną diagnostyką. Fold 4 wybierał checkpoint; fold 0 był już
oglądany i zawiera jedną dużą grupę wizualną. **Nie są to nowe niezależne testy.**
Orientacja maski pęczków nie jest etykietą eksperckiego PA.


In [1]:
from pathlib import Path
import os, sys, json, hashlib, importlib.util, subprocess, base64, zlib
PROJECT = Path('/content/drive/MyDrive/UMUD_Challenge')
APO_CHECKPOINT = Path('/content/drive/MyDrive/UMUD_R12B/best_dinov2_frozen.pt')
DO_ORIENTATION = True  # Pozostaw True: potrzebne do decyzji o kolejnym treningu.
if not Path('/content/drive/MyDrive').is_dir():
    from google.colab import drive
    drive.mount('/content/drive')
if not PROJECT.is_dir(): raise FileNotFoundError(PROJECT)
R17 = PROJECT/'outputs/R17_v1_0_0'
OUT = PROJECT/'outputs/R20_v1_0_0'
OUT.mkdir(parents=True, exist_ok=True)
missing=[]
for module, package in [('numpy','numpy'),('pandas','pandas'),('cv2','opencv-python-headless'),
                        ('scipy','scipy'),('skimage','scikit-image'),('timm','timm')]:
    if importlib.util.find_spec(module) is None: missing.append(package)
if missing: subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
import torch
if not torch.cuda.is_available(): raise RuntimeError('Włącz GPU w ustawieniach Colaba i uruchom ponownie.')
DEVICE='cuda'
print('GPU:',torch.cuda.get_device_name(0)); print('Wyniki:',OUT)


Mounted at /content/drive
GPU: Tesla T4
Wyniki: /content/drive/MyDrive/UMUD_Challenge/outputs/R20_v1_0_0


In [2]:
PAYLOAD='...'
raw=zlib.decompress(base64.b64decode(PAYLOAD))
if hashlib.sha256(raw).hexdigest()!='f2dca1cd64ce855538575b8d1f4adb415b3d448be405c3c070b01867e0e19fb9': raise ValueError('Uszkodzony pakiet R20')
bundle=json.loads(raw); PACKAGE=OUT/'package'
for name,content in bundle['files'].items():
    if hashlib.sha256(content.encode()).hexdigest()!=bundle['sha256'][name]: raise ValueError(name)
    path=PACKAGE/name; path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(content,encoding='utf-8')
    if hashlib.sha256(path.read_bytes()).hexdigest()!=bundle['sha256'][name]: raise IOError(path)
sys.path.insert(0,str(PACKAGE))
for name in ('r20_run','r20_geometry','umud_model','umud_core'): sys.modules.pop(name,None)
from umud_core import sha256, atomic_json, read_gray
from umud_model import load_model, load_legacy_apo, predict_full, predict_legacy_apo, seed_all
from r20_geometry import CONFIG, selftest, check_previous, digest
from r20_run import run_geometry, run_orientation, make_reports, gallery, export_report
import numpy as np, pandas as pd, cv2, scipy, skimage, timm
ASSETS=PACKAGE/'assets'
PROVENANCE=json.loads((PACKAGE/'PROVENANCE.json').read_text())
SELFTEST=selftest(); atomic_json(OUT/'SELFTEST.json',SELFTEST); print(SELFTEST)


{'pass': True, 'straight_PA': 30.004681504849913, 'curved_local_PA': 17.474164651091726, 'checks': ['baseline_identity', 'mirror', 'units', 'isotropic_resize', 'unsupported_intersection', 'missing_apo', 'analytic_curve']}


## 1. Kontrakt wejściowy i wagi


In [3]:
CHECKPOINTS={'refit':R17/'refit/best.pt','develop':R17/'develop/best.pt','apo':APO_CHECKPOINT}
for kind in ('refit','apo')+ (('develop',) if DO_ORIENTATION else ()):
    path=CHECKPOINTS[kind]
    if not path.is_file(): raise FileNotFoundError(path)
    if sha256(path)!=PROVENANCE['checkpoints'][kind]: raise ValueError('Niewłaściwe wagi: '+kind)
for kind in ('refit',)+ (('develop',) if DO_ORIENTATION else ()):
    selected=json.loads((R17/kind/'selection.json').read_text())
    if selected['checkpoint_sha256']!=PROVENANCE['checkpoints'][kind] or float(selected['threshold'])!=.65:
        raise ValueError('Niewłaściwy kontrakt selection.json: '+kind)
anchor=pd.read_csv(ASSETS/'anchor_039776.csv',dtype=str)
registry=pd.read_csv(ASSETS/'scale_registry.csv')
previous=pd.read_csv(ASSETS/'refit_test_measurements.csv')
assert len(anchor)==309 and anchor.image_id.is_unique
assert set(anchor.image_id)==set(registry.image_id)==set(previous.image_id)
registry=registry.set_index('image_id').loc[anchor.image_id]
previous=previous.set_index('image_id').loc[anchor.image_id]
test_manifest=pd.DataFrame({'image_id':anchor.image_id.to_list(),
    'path':[str(PROJECT/'data/competition_test_v1'/name) for name in anchor.image_id],
    'px_per_mm':registry.SCALE_PX_PER_CM_FINAL.to_numpy(float)/10,
    'image_sha256':previous.image_sha256.to_list()})
truth=pd.read_csv(ASSETS/'external_truth.csv')
external_old=pd.read_csv(ASSETS/'refit_external_measurements.csv').set_index('image_id').loc[truth.image_id]
def project_path(value):
    original=Path('/content/drive/MyDrive/UMUD_Challenge')
    return str(PROJECT/Path(value).relative_to(original))
external_manifest=pd.DataFrame({'image_id':truth.image_id,'path':[project_path(v) for v in truth.path],
    'px_per_mm':truth.px_per_cm.to_numpy(float)/10,'image_sha256':external_old.image_sha256.to_list()})
samples=pd.DataFrame(json.loads((ASSETS/'diagnostic_samples.json').read_text()))
samples['image']=[str(PROJECT/'data/competition_train_v1/fasc_images'/v) for v in samples.file_name]
samples['mask_paths']=[json.dumps([str(PROJECT/'data/competition_train_v1/fasc_masks'/v) for v in names])
                       for names in samples.annotation_names]
assert len(samples)==824 and samples.sample_id.is_unique and samples.groupby('group_id').fold.nunique().max()==1
paths=test_manifest.path.tolist()+external_manifest.path.tolist()
if DO_ORIENTATION:
    paths+=samples.image.tolist()+[p for x in samples.mask_paths for p in json.loads(x)]
missing=[p for p in paths if not Path(p).is_file()]
if missing: raise FileNotFoundError(f'Brak {len(missing)} plików, pierwsze: {missing[:8]}')
for name,manifest in [('test',test_manifest),('OSF',external_manifest)]:
    assert manifest.image_id.is_unique and np.isfinite(manifest.px_per_mm).all() and (manifest.px_per_mm>0).all()
    print(name,len(manifest),'plików obecnych; hashe obrazów sprawdzane przed każdą inferencją.')
versions={k:str(v.__version__) for k,v in [('torch',torch),('numpy',np),('pandas',pd),('cv2',cv2),
                                         ('scipy',scipy),('skimage',skimage),('timm',timm)]}
code_hashes={name:sha256(PACKAGE/name) for name in ('umud_core.py','umud_model.py','r20_geometry.py','r20_run.py')}
INFER_SIG={'fasc_sha256':PROVENANCE['checkpoints']['refit'],'apo_sha256':PROVENANCE['checkpoints']['apo'],
           'thresholds':[.5,.65],'tta':False,'side':768,'versions':versions,
           'code':{k:code_hashes[k] for k in ('umud_core.py','umud_model.py')}}
GEOM_SIG={'config':CONFIG,'code':code_hashes,'versions':versions}
RUN_CONFIG={'version':'R20_v1.0.0','inference':INFER_SIG,'geometry':GEOM_SIG,'do_orientation':DO_ORIENTATION,
            'anchor_sha256':sha256(ASSETS/'anchor_039776.csv'),'checkpoint_hashes':PROVENANCE['checkpoints'],
            'orientation_native_pixels':True,'no_new_holdout':True}
if (OUT/'RUN_CONFIG.json').exists():
    old=json.loads((OUT/'RUN_CONFIG.json').read_text())
    if old!=RUN_CONFIG:
        raise RuntimeError('Konfiguracja/środowisko zmieniły się. Ustaw nowy katalog OUT, aby rozdzielić przebiegi.')
atomic_json(OUT/'RUN_CONFIG.json',RUN_CONFIG)
atomic_json(OUT/'PREFLIGHT.json',{'test_images':309,'osf_images':35,'orientation_images':len(samples),
                              'source_files_present':len(paths),'weights_checked':True,'fold_counts':{str(k):int(v) for k,v in samples.groupby('fold').size().items()}})
seed_all(20260918); torch.backends.cudnn.benchmark=False
print('Preflight OK. Zaczynamy od trzech obrazów kontrolnych R17.')


test 309 plików obecnych; hashe obrazów sprawdzane przed każdą inferencją.
OSF 35 plików obecnych; hashe obrazów sprawdzane przed każdą inferencją.
Preflight OK. Zaczynamy od trzech obrazów kontrolnych R17.


## 2. Zamrożone maski refit: test i OSF


In [4]:
fasc_model,fasc_meta=load_model(CHECKPOINTS['refit'],DEVICE)
if fasc_meta['config']['side']!=768: raise ValueError('Niepoprawny rozmiar wejścia checkpointu')
apo_model=load_legacy_apo(CHECKPOINTS['apo'],DEVICE)
def refit_predictor(path):
    gray=read_gray(path)
    return predict_legacy_apo(apo_model,path,DEVICE),predict_full(fasc_model,gray,768,DEVICE,tta=False)
canary,_=run_geometry(test_manifest.head(3),refit_predictor,OUT,INFER_SIG,GEOM_SIG,'canary')
expected=pd.read_csv(ASSETS/'refit_test_measurements.csv').head(3)
canary_report=check_previous(canary,expected); atomic_json(OUT/'CANARY_REPRODUCTION.json',canary_report)
if not canary_report['pass']:
    raise RuntimeError('Kontrola R17 nie przeszła; zobacz CANARY_REPRODUCTION.json. Nie porównuj nowych wariantów przed wyjaśnieniem różnic.')
test,test_fragments=run_geometry(test_manifest,refit_predictor,OUT,INFER_SIG,GEOM_SIG,'test')
external,external_fragments=run_geometry(external_manifest,refit_predictor,OUT,INFER_SIG,GEOM_SIG,'external')
del fasc_model,apo_model
import gc;gc.collect();torch.cuda.empty_cache()
display(test[['image_id','pa_r17_median','pa_coherent_weighted','pa_local_median','local_coverage']].head())


canary 3 / 3
test 20 / 309
test 40 / 309
test 60 / 309
test 80 / 309
test 100 / 309
test 120 / 309
test 140 / 309
test 160 / 309
test 180 / 309
test 200 / 309
test 220 / 309
test 240 / 309
test 260 / 309
test 280 / 309
test 300 / 309
test 309 / 309
external 20 / 35
external 35 / 35


,image_id,pa_r17_median,pa_coherent_weighted,pa_local_median,local_coverage
0,IMG_00001.tif,17.204451,17.122494,18.504761,0.333333
1,IMG_00002.tif,9.678399,10.238921,11.690350,0.321429
2,IMG_00003.tif,13.224084,13.468777,11.133952,0.366667
3,IMG_00004.tif,11.683812,11.918266,14.972301,0.173913
4,IMG_00005.tif,19.755129,19.905212,19.778201,0.391304


## 3. Błąd orientacji względem anotacji — tylko develop


In [5]:
orientation=None
if DO_ORIENTATION:
    develop,develop_meta=load_model(CHECKPOINTS['develop'],DEVICE)
    assert develop_meta['config']['side']==768
    def develop_predictor(path):
        return predict_full(develop,read_gray(path),768,DEVICE,tta=False)
    orientation_signature={'checkpoint_sha256':PROVENANCE['checkpoints']['develop'],
                           'code':code_hashes,'versions':versions,'threshold':.65,'side':768,
                           'registration':'STRETCH_FULL_NORMALIZED_COORDINATES',
                           'metrics':'native orientation proxy; native center tolerance2px; target dilation1px'}
    orientation=run_orientation(samples,develop_predictor,OUT,orientation_signature)
    del develop;gc.collect();torch.cuda.empty_cache()
    display(pd.read_csv(OUT/'orientation_group_summary.csv').groupby('fold').agg(
        images=('n','sum'),groups=('group_id','size'),group_proxy_MAE=('orientation_proxy_MAE','mean')))
else:
    print('Pominięto orientację GT. Raport nie wystarczy do oceny potrzeby treningu segmentera.')


develop orientation 25 / 824
develop orientation 50 / 824
develop orientation 75 / 824
develop orientation 100 / 824
develop orientation 125 / 824
develop orientation 150 / 824
develop orientation 175 / 824
develop orientation 200 / 824
develop orientation 225 / 824
develop orientation 250 / 824
develop orientation 275 / 824
develop orientation 300 / 824
develop orientation 325 / 824
develop orientation 350 / 824
develop orientation 375 / 824
develop orientation 400 / 824
develop orientation 425 / 824
develop orientation 450 / 824
develop orientation 475 / 824
develop orientation 500 / 824
develop orientation 525 / 824
develop orientation 550 / 824
develop orientation 575 / 824
develop orientation 600 / 824
develop orientation 625 / 824
develop orientation 650 / 824
develop orientation 675 / 824
develop orientation 700 / 824
develop orientation 725 / 824
develop orientation 750 / 824
develop orientation 775 / 824
develop orientation 800 / 824
develop orientation 824 / 824


,images,groups,group_proxy_MAE
fold,,,
0,370,1,5.472629
4,454,240,1.093453


## 4. Raport, przykłady i paczka wyników


In [6]:
summary,osf_comparison=make_reports(test,external,truth,orientation,ASSETS,OUT)
gallery(test_manifest,test,OUT,INFER_SIG,maximum=12)
result_zip=export_report(OUT)
print(json.dumps(summary,ensure_ascii=False,indent=2))
display(osf_comparison)
from IPython.display import FileLink,display
display(FileLink(str(result_zip)))
print('GOTOWE:',result_zip)
print('Udostępnij UMUD_R20_WYNIKI.zip lub pozostaw go na Drive do analizy.')
if not summary['baseline_reproduction_pass']:
    print('UWAGA: pełne odtworzenie R17 nie przeszło. Najpierw analizujemy baseline_reproduction.json.')


{
  "version": "R20_v1.0.0",
  "anchor_public_score": 0.39776,
  "anchor_sha256": "115408e915674e3433ed59a7a734542afd7764ff4d3ec543de7c58050b72d929",
  "baseline_reproduction_pass": true,
  "test_images": 309,
  "osf_images": 35,
  "orientation": [
    {
      "fold": 0,
      "images": 370,
      "groups": 1,
      "images_with_angle": 368,
      "image_weighted_proxy_MAE": 5.472629161439466,
      "group_weighted_proxy_MAE": 5.472629161439466,
      "warning": "Fold 0 previously inspected, one group; fold 4 used for selection. No new holdout."
    },
    {
      "fold": 4,
      "images": 454,
      "groups": 240,
      "images_with_angle": 454,
      "image_weighted_proxy_MAE": 1.026980955407515,
      "group_weighted_proxy_MAE": 1.093453419588179,
      "warning": "Fold 0 previously inspected, one group; fold 4 used for selection. No new holdout."
    }
  ],
  "local_test_image_coverage": 295,
  "osf_role": "repeatedly_used_diagnostic_not_model_selection_or_LB_forecast",
  "scalar_

,variant,reference,support,n,total,candidate_available,candidate_MAE,reference_MAE,paired_delta_MAE,improved,worsened,role
0,pa_r17_median,pa_r17_median,pairwise,34,35,34,1.513924,1.513924,0.000000,0,0,REUSED_OSF_DIAGNOSTIC_ONLY
1,pa_r17_median,pa_r17_median,all_variants_common,33,35,34,1.372271,1.372271,0.000000,0,0,REUSED_OSF_DIAGNOSTIC_ONLY
2,pa_center_median,pa_r17_median,pairwise,34,35,34,1.626211,1.513924,0.112288,11,23,REUSED_OSF_DIAGNOSTIC_ONLY
3,pa_center_median,pa_r17_median,all_variants_common,33,35,34,1.450320,1.372271,0.078049,11,22,REUSED_OSF_DIAGNOSTIC_ONLY
4,pa_length_weighted,pa_r17_median,pairwise,34,35,34,1.382458,1.513924,-0.131466,14,14,REUSED_OSF_DIAGNOSTIC_ONLY
5,pa_length_weighted,pa_r17_median,all_variants_common,33,35,34,1.232498,1.372271,-0.139773,14,13,REUSED_OSF_DIAGNOSTIC_ONLY
6,pa_coherent_weighted,pa_r17_median,pairwise,34,35,34,1.382458,1.513924,-0.131466,14,14,REUSED_OSF_DIAGNOSTIC_ONLY
7,pa_coherent_weighted,pa_r17_median,all_variants_common,33,35,34,1.232498,1.372271,-0.139773,14,13,REUSED_OSF_DIAGNOSTIC_ONLY
8,pa_local_median,pa_r17_median,pairwise,33,35,33,1.253922,1.372271,-0.118349,16,17,REUSED_OSF_DIAGNOSTIC_ONLY
9,pa_local_median,pa_r17_median,all_variants_common,33,35,33,1.253922,1.372271,-0.118349,16,17,REUSED_OSF_DIAGNOSTIC_ONLY


/content/drive/MyDrive/UMUD_Challenge/outputs/R20_v1_0_0/UMUD_R20_WYNIKI.zip

GOTOWE: /content/drive/MyDrive/UMUD_Challenge/outputs/R20_v1_0_0/UMUD_R20_WYNIKI.zip
Udostępnij UMUD_R20_WYNIKI.zip lub pozostaw go na Drive do analizy.


## Jak wybrać następny ruch

Nie wybieramy automatycznie minimum MAE z OSF: ten zbiór źle przewidział ranking FL.
Sprawdzamy zgodność baseline, pokrycie, wynik na wspólnych obrazach i fragmentach,
błąd proxy w grupach oraz nakładki. Z tych danych wybieramy **jedną** zmianę:
referencja rozcięgna, agregacja pęczków albo nadzór orientacji w kolejnym treningu.

Pełne rozdzielenie błędu PA wymaga spójnie sparowanych masek rozcięgien, pęczków
i eksperckiego scalar PA. Ten notebook ich nie wymyśla i nie łączy gałęzi po samej
nazwie pliku. Diagnostyka GT dotyczy orientacji pęczków; geometria ma kontrolę
syntetyczną i pomiary scalar OSF. Dla lokalnej stycznej sprawdzamy osobno efekt
zmiany referencji i efekt odrzucenia fragmentów bez obserwowanego przecięcia.

Gdy diagnoza wskaże błąd segmentera, kolejny trening powinien dać wspólne,
grupowe predykcje OOF dla PA/FL/MT. Same kolejne wagi kalibracji leaderboardowej
nie zastępują walidacji generalizacji. Aktualna kotwica 0,39776 zostaje zachowana.
